## **Installs**

In [ ]:
!pip install -q -U transformers

In [2]:
import transformers
print(transformers.__version__)

5.14.1


## **Hugging Face Token**

In [3]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("my_hf_token")

In [4]:
from huggingface_hub import login
login(token=hf_token)

## **Imports**

In [5]:
import json
import re
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import gc, traceback

In [6]:
MODEL_NAME = "mistralai/Mistral-Nemo-Instruct-2407"
QUESTIONS_PATH = "/kaggle/input/datasets/anarvaaa/questions/new_research_questions.json"  # update path
TEMPLATE_PATH = "/kaggle/input/datasets/anarvaaa/questions/prompt_template.txt"           # update path
OUTPUT_PATH = "/kaggle/working/pico_outputs.json"
HF_TOKEN = hf_token

In [7]:
print("GPUs visible:", torch.cuda.device_count()) 

GPUs visible: 2


## **Load Model**

In [10]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN,fix_mistral_regex=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    dtype=torch.float16,      # NOT torch_dtype (deprecated, as you already found)
    device_map="auto",
)
model.eval()

Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(131072, 5120)
    (layers): ModuleList(
      (0-39): 40 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear(in_features=5120, out_features=4096, bias=False)
          (k_proj): Linear(in_features=5120, out_features=1024, bias=False)
          (v_proj): Linear(in_features=5120, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=5120, bias=False)
        )
        (mlp): MistralMLP(
          (gate_proj): Linear(in_features=5120, out_features=14336, bias=False)
          (up_proj): Linear(in_features=5120, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=5120, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): MistralRMSNorm((5120,), eps=1e-05)
        (post_attention_layernorm): MistralRMSNorm((5120,), eps=1e-05)
      )
    )
    (norm): MistralRMSNorm((5120,

In [11]:
print(model.hf_device_map)

{'model.embed_tokens': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 0, 'model.layers.6': 0, 'model.layers.7': 0, 'model.layers.8': 0, 'model.layers.9': 0, 'model.layers.10': 0, 'model.layers.11': 0, 'model.layers.12': 0, 'model.layers.13': 0, 'model.layers.14': 0, 'model.layers.15': 0, 'model.layers.16': 0, 'model.layers.17': 0, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.layers.26': 1, 'model.layers.27': 1, 'model.layers.28': 1, 'model.layers.29': 1, 'model.layers.30': 1, 'model.layers.31': 1, 'model.layers.32': 1, 'model.layers.33': 1, 'model.layers.34': 1, 'model.layers.35': 1, 'model.layers.36': 1, 'model.layers.37': 1, 'model.layers.38': 1, 'model.layers.39': 1, 'model.norm': 1, 'model.rotary_emb': 1, 'lm_head': 1}


## **Load Template and Questions**

In [12]:
with open(TEMPLATE_PATH, "r") as f:
    FEW_SHOT_TEMPLATE = f.read().strip()

with open(QUESTIONS_PATH, "r") as f:
    questions = json.load(f)

## **Helper Functions**

In [13]:
def build_prompt(question_text: str) -> str:
    return f"{FEW_SHOT_TEMPLATE}\n\nQuestion: {question_text}\n\nP:"

def extract_pico(generated_text: str) -> str:
    text = "P:" + generated_text.strip()
    match = re.search(r"(P:.*?O:.*?)(?:\n\n|\nQuestion:|\nExample|\Z)", text, re.DOTALL)
    return match.group(1).strip() if match else text.strip()

def generate_pico(question_text: str, max_new_tokens: int = 150) -> str:
    prompt = build_prompt(question_text)
    messages = [
        {"role": "system", "content": "You output ONLY the PICO breakdown in the exact format shown (P:, I:, C:, O: lines). Never add explanations, preambles, or commentary."},
        {"role": "user", "content": prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)

    eos_id = tokenizer.eos_token_id
    if isinstance(eos_id, list):
        eos_id = eos_id[0]

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,               # <-- unpack input_ids + attention_mask directly
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=eos_id,
        )

    generated = output_ids[0][inputs["input_ids"].shape[-1]:]
    decoded = tokenizer.decode(generated, skip_special_tokens=True)

    del inputs, output_ids, generated
    gc.collect()
    torch.cuda.empty_cache()

    return extract_pico(decoded)

## **Run Code**

In [14]:
results = []

for i, item in enumerate(questions):
    q_text = item["question"]
    print(f"[{i+1}/{len(questions)}] id={item['id']} | {q_text[:70]}...")

    try:
        pico_output = generate_pico(q_text)
    except Exception as e:
        print(f"  !! Error on id={item['id']}: {repr(e)}")
        traceback.print_exc()
        pico_output = f"ERROR: {repr(e)}"

    results.append({
        "id": item["id"],
        "set": item["set"],
        "serial_number": item["serial_number"],
        "section": item["section"],
        "question": q_text,
        "pico_output": pico_output,
    })

    # Save progress every 10 questions (cheap insurance against Kaggle timeouts)
    if (i + 1) % 10 == 0 or (i + 1) == len(questions):
        with open(OUTPUT_PATH, "w") as f:
            json.dump(results, f, indent=2)

print(f"\nDone. Saved {len(results)} results to {OUTPUT_PATH}")

[1/200] id=1 | In adults with resistant hypertension, does renal denervation compared...
[2/200] id=2 | In adults with HFrEF (LVEF ≤40%), does vericiguat compared with placeb...
[3/200] id=3 | In adults post-STEMI, does low-dose colchicine compared with placebo r...
[4/200] id=4 | In low-surgical-risk adults with severe aortic stenosis, does transcat...
[5/200] id=5 | In adults with atrial fibrillation and moderate CKD, does apixaban com...
[6/200] id=6 | In adults with familial hypercholesterolemia, does twice-yearly inclis...
[7/200] id=7 | In adults with symptomatic peripheral artery disease, does supervised ...
[8/200] id=8 | In adults with cardiogenic shock, does early mechanical circulatory su...
[9/200] id=9 | In adults with obstructive hypertrophic cardiomyopathy, does mavacamte...
[10/200] id=10 | In adults with recurrent pericarditis, does rilonacept compared with s...
[11/200] id=11 | In adults with relapsing-remitting multiple sclerosis, does early high...
[12/200] id=12 | 